In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder.appName("Handling_Date_Type").getOrCreate()

26/08/31 11:59:55 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [5]:
#Create csv file 

csv_data = """id,date_iso,date_dmy,date_mdy,timestamp
1,2023-01-15,15/01/2023,01/15/2023,2023-01-15 10:30:00
2,2023-05-20,20/05/2023,05/20/2023,2023-05-20 15:45:00
3,InvalideDate,31/02/2023,02/31/2023,InvalideTimestamp
4,,,,

"""
#Save the CSV file

with open("dates_data.csv","w") as f:
    f.write(csv_data)

In [6]:
!ls *dates_data.csv*

dates_data.csv


In [7]:
!hadoop fs -put dates_data.csv /data/dates_data.csv

put: `/data/dates_data.csv': File exists


In [8]:
!hadoop fs -ls  /data/

Found 7 items
-rw-r--r--   2 root hadoop        344 2026-08-26 17:17 /data/customers_with_errors.csv
-rw-r--r--   2 root hadoop        212 2026-08-28 14:19 /data/dates_data.csv
-rw-r--r--   2 root hadoop       5488 2026-08-25 13:20 /data/first_100_customers.csv
drwxr-xr-x   - root hadoop          0 2026-08-27 08:48 /data/write_output.csv
drwxr-xr-x   - root hadoop          0 2026-08-27 09:03 /data/write_output_1.csv
drwxr-xr-x   - root hadoop          0 2026-08-27 09:06 /data/write_output_2.csv
drwxr-xr-x   - root hadoop          0 2026-08-27 09:07 /data/write_output_3.csv


In [36]:
ddl_schema ="""
    id INT,
    date_iso Date,
    date_dmy Date,
    date_mdy Date ,
    timestamp TIMESTAMP
"""
from pyspark.sql.types import StructField ,StringType,IntegerType,StringType,StructType
#StructType for the schema

struct_schema = StructType([
    StructField("id",IntegerType(),nullable=True),
    StructField("date_iso",StringType(),nullable=True),
    StructField("date_dmy",StringType(),nullable=True),
    StructField("date_mdy",StringType(),nullable=True),
    StructField("timestamp",StringType(),nullable=True),
])

#Read the csv file into DataFrame 
df_file = spark.read.option("header",True).schema(ddl_schema).csv("/data/dates_data.csv")

#Show the DataFrame
df_file.show(truncate=False)

+---+----------+--------+--------+-------------------+
|id |date_iso  |date_dmy|date_mdy|timestamp          |
+---+----------+--------+--------+-------------------+
|1  |2023-01-15|NULL    |NULL    |2023-01-15 10:30:00|
|2  |2023-05-20|NULL    |NULL    |2023-05-20 15:45:00|
|3  |NULL      |NULL    |NULL    |NULL               |
|4  |NULL      |NULL    |NULL    |NULL               |
+---+----------+--------+--------+-------------------+



In [37]:
df_file.printSchema()

root
 |-- id: integer (nullable = true)
 |-- date_iso: date (nullable = true)
 |-- date_dmy: date (nullable = true)
 |-- date_mdy: date (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [44]:
#Sample data with multiple date formats
data = [
    (1,"2023-01-15","15/01/2023","01/15/2023","2023-01-15 10:30:00"),
    (2,"2023-05-20","20/05/2023","05/20/2023","2023-05-20 15:45:00"),
    (3,"InvalideDate","31/02/2023","02/31/2023","InvalideTimestamp"), #Invalid dates
    (4,None,None,None,None) # Null values
]
#Define  column names

columns =["id","date_iso","date_dmy","date_mdy","timestamp"]

#Create DataFrame
df = spark.createDataFrame(data,schema=columns)

#Show the DataFrame

df.show(truncate=False)


+---+------------+----------+----------+-------------------+
|id |date_iso    |date_dmy  |date_mdy  |timestamp          |
+---+------------+----------+----------+-------------------+
|1  |2023-01-15  |15/01/2023|01/15/2023|2023-01-15 10:30:00|
|2  |2023-05-20  |20/05/2023|05/20/2023|2023-05-20 15:45:00|
|3  |InvalideDate|31/02/2023|02/31/2023|InvalideTimestamp  |
|4  |NULL        |NULL      |NULL      |NULL               |
+---+------------+----------+----------+-------------------+



In [45]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- date_iso: string (nullable = true)
 |-- date_dmy: string (nullable = true)
 |-- date_mdy: string (nullable = true)
 |-- timestamp: string (nullable = true)



In [46]:
from pyspark.sql.functions import to_date
df = df\
        .withColumn('parsed_date_iso',to_date(df.date_iso,'yyyy-MM-dd'))\
        .withColumn('parsed_date_dmy',to_date(df.date_dmy,'dd/MM/yyyy'))\
        .withColumn('parsed_date_mdy',to_date(df.date_mdy,'MM/dd/yyyy'))

In [47]:
df.show(truncate=False)

+---+------------+----------+----------+-------------------+---------------+---------------+---------------+
|id |date_iso    |date_dmy  |date_mdy  |timestamp          |parsed_date_iso|parsed_date_dmy|parsed_date_mdy|
+---+------------+----------+----------+-------------------+---------------+---------------+---------------+
|1  |2023-01-15  |15/01/2023|01/15/2023|2023-01-15 10:30:00|2023-01-15     |2023-01-15     |2023-01-15     |
|2  |2023-05-20  |20/05/2023|05/20/2023|2023-05-20 15:45:00|2023-05-20     |2023-05-20     |2023-05-20     |
|3  |InvalideDate|31/02/2023|02/31/2023|InvalideTimestamp  |NULL           |NULL           |NULL           |
|4  |NULL        |NULL      |NULL      |NULL               |NULL           |NULL           |NULL           |
+---+------------+----------+----------+-------------------+---------------+---------------+---------------+



 # Timestamp

In [53]:
from pyspark.sql.functions import to_timestamp,year,month,dayofmonth,hour,minute,day

In [54]:
df = df.withColumn('parsed_timestamp',to_timestamp(df.timestamp))

In [55]:
df.show()

+---+------------+----------+----------+-------------------+---------------+---------------+---------------+-------------------+
| id|    date_iso|  date_dmy|  date_mdy|          timestamp|parsed_date_iso|parsed_date_dmy|parsed_date_mdy|   parsed_timestamp|
+---+------------+----------+----------+-------------------+---------------+---------------+---------------+-------------------+
|  1|  2023-01-15|15/01/2023|01/15/2023|2023-01-15 10:30:00|     2023-01-15|     2023-01-15|     2023-01-15|2023-01-15 10:30:00|
|  2|  2023-05-20|20/05/2023|05/20/2023|2023-05-20 15:45:00|     2023-05-20|     2023-05-20|     2023-05-20|2023-05-20 15:45:00|
|  3|InvalideDate|31/02/2023|02/31/2023|  InvalideTimestamp|           NULL|           NULL|           NULL|               NULL|
|  4|        NULL|      NULL|      NULL|               NULL|           NULL|           NULL|           NULL|               NULL|
+---+------------+----------+----------+-------------------+---------------+---------------+-----

In [56]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- date_iso: string (nullable = true)
 |-- date_dmy: string (nullable = true)
 |-- date_mdy: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- parsed_date_iso: date (nullable = true)
 |-- parsed_date_dmy: date (nullable = true)
 |-- parsed_date_mdy: date (nullable = true)
 |-- parsed_timestamp: timestamp (nullable = true)



In [57]:
df = df\
.withColumn('year',year(df.parsed_timestamp))\
.withColumn('month',month(df.parsed_timestamp))\
.withColumn('day',day(df.parsed_timestamp))\
.withColumn('hour',hour(df.parsed_timestamp))\
.withColumn('minute',minute(df.parsed_timestamp))
df.show()

+---+------------+----------+----------+-------------------+---------------+---------------+---------------+-------------------+----+-----+----+----+------+
| id|    date_iso|  date_dmy|  date_mdy|          timestamp|parsed_date_iso|parsed_date_dmy|parsed_date_mdy|   parsed_timestamp|year|month| day|hour|minute|
+---+------------+----------+----------+-------------------+---------------+---------------+---------------+-------------------+----+-----+----+----+------+
|  1|  2023-01-15|15/01/2023|01/15/2023|2023-01-15 10:30:00|     2023-01-15|     2023-01-15|     2023-01-15|2023-01-15 10:30:00|2023|    1|  15|  10|    30|
|  2|  2023-05-20|20/05/2023|05/20/2023|2023-05-20 15:45:00|     2023-05-20|     2023-05-20|     2023-05-20|2023-05-20 15:45:00|2023|    5|  20|  15|    45|
|  3|InvalideDate|31/02/2023|02/31/2023|  InvalideTimestamp|           NULL|           NULL|           NULL|               NULL|NULL| NULL|NULL|NULL|  NULL|
|  4|        NULL|      NULL|      NULL|               NUL

In [58]:
from pyspark.sql.functions import datediff

df = df.withColumn('days_difference',datediff(df.parsed_date_mdy,df.parsed_date_iso))
df.select('parsed_date_mdy','parsed_date_iso','days_difference').show(truncate=False)

+---------------+---------------+---------------+
|parsed_date_mdy|parsed_date_iso|days_difference|
+---------------+---------------+---------------+
|2023-01-15     |2023-01-15     |0              |
|2023-05-20     |2023-05-20     |0              |
|NULL           |NULL           |NULL           |
|NULL           |NULL           |NULL           |
+---------------+---------------+---------------+



In [65]:
spark.stop()

ConnectionRefusedError: [Errno 111] Connection refused